### 2. Import necessary libraries

In [2]:
try:
    import datasets, evaluate, accelerate
    import gradio as gr
except ModuleNotFoundError:
    !pip install -U datasets evaluate accelerate gradio
    import datasets, evaluate, accelerate
    import gradio as gr
import random
import numpy as np
import pandas as pd
import torch
import transformers

print(f"Using transformers version: {transformers.__version__}")
print(f"Using torch version: {torch.__version__}")
print(f"Using datasets version: {datasets.__version__}")

/Users/mdashikadnan/Documents/adnanedu/python/ztm/hugging_face_custom_ai_model/code-repo/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using transformers version: 4.45.2
Using torch version: 2.4.1
Using datasets version: 3.0.1


## 3. Getting a dataset
Building food not food text classification model: need food not food text dataset.

In [32]:
from datasets import load_dataset

# Load the dataset from Hugging Face Hub
dataset = load_dataset(path="mrdbourke/learn_hf_food_not_food_image_captions")

#Inspect the dataset
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 250
    })
})

In [33]:
# What features are there?
dataset.column_names

{'train': ['text', 'label']}

In [34]:
# Access the training split
dataset["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 250
})

In [35]:
# How about we check out a single sample?
# We can do so with indexing.
dataset["train"][0]

{'text': 'Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
 'label': 'food'}

### Inspect random samples

In [36]:
import random

random_indexs = random.sample(range(len(dataset["train"])), 5)
print(random_indexs)

random_samples = dataset["train"][random_indexs]

print(f"[INFO] Random samples from dataset:\n")
for text,label in zip(random_samples["text"], random_samples["label"]):
    print(f"Text: {text} | Label: {label}")

[137, 199, 78, 126, 218]
[INFO] Random samples from dataset:

Text: A whole papaya on a white plate with two apples on the side | Label: food
Text: Crunchy sushi roll with a creamy filling, featuring shrimp tempura and avocado. | Label: food
Text: Low-carb sushi roll with cucumber or seaweed wraps instead of rice. | Label: food
Text: Set of tea towels folded in a kitchen | Label: not_food
Text: Mouthwatering mushroom curry, featuring shiitake and button mushrooms in a rich coconut milk sauce with spices and herbs. | Label: food


In [37]:
# Get unique label values
dataset["train"].unique("label")

['food', 'not_food']

In [38]:
# Check the count of each label
from collections import Counter
Counter(dataset["train"]["label"])

Counter({'food': 125, 'not_food': 125})

In [39]:
# Turn our dataset into a DataFrame and get a random sample
food_not_food_df = pd.DataFrame(dataset["train"])
food_not_food_df.sample(7)

,text,label
243,"Relaxing on the porch, a couple enjoys the com...",not_food
175,Set of pillows arranged on a couch,not_food
172,"Spicy prawn curry with fresh mint garnish, fea...",food
161,Set of measuring cups nested in a drawer,not_food
230,Wooden cutting board with a chef's knife ready...,not_food
242,"Fennel in a bowl, sprinkled with lemon zest an...",food
2,"Watching TV together, a family has their dog s...",not_food


In [40]:
food_not_food_df["label"].value_counts()

label
food        125
not_food    125
Name: count, dtype: int64

## 4. Preparing data for text classification

1. Tokenization - turning our text into a numerical representation (machines prefer numbers rather than words), for example, {"a": 0, "b": 1, "c": 2...}.
2. Creating a train/test split - right now our data is in a training split only but we'll create a test set to evaluate our model's performance.

In [41]:
# Create a mapping for labels to numeric value
id2label = {0: "not_food", 1: "food"}
label2id = {"not_food": 0, "food": 1}

print(id2label)
print(label2id)

{0: 'not_food', 1: 'food'}
{'not_food': 0, 'food': 1}


In [42]:
# Create mappings programmatically from dataset
id2label = {idx: label for idx, label in enumerate(dataset["train"].unique("label")[::-1])}
label2id = {label: idx for idx, label in id2label.items()}
print(id2label)
print(label2id)

{0: 'not_food', 1: 'food'}
{'not_food': 0, 'food': 1}


In [44]:
id2label = {}
for idx, label in enumerate(dataset["train"].unique("label")[::-1]):
    print(idx, label)
    id2label[idx] = label

0 not_food
1 food


In [45]:
# Turn labels into 0 or 1
def map_labels_to_number(example):
    example["label"] = label2id[example["label"]]
    return example

example_sample = {"text": "This is a sentence about my favourite food: honey", "label": "food"}

# Test our function
print(map_labels_to_number(example_sample))

{'text': 'This is a sentence about my favourite food: honey', 'label': 1}


In [46]:
# Map our dataset labels to numbers (the whole thing)
# We do this with dataset.map()
dataset = dataset["train"].map(map_labels_to_number)
dataset[:5]

Map: 100%|██████████| 250/250 [00:00<00:00, 25249.86 examples/s]


{'text': ['Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
  'Set of books stacked on a desk',
  'Watching TV together, a family has their dog stretched out on the floor',
  'Wooden dresser with a mirror reflecting the room',
  'Lawn mower stored in a shed'],
 'label': [1, 0, 0, 0, 0]}

In [48]:
# Shuffle data and look at 5 more random samples
dataset.shuffle()[:5]

{'text': ['Set of curtains draped over a window',
  'Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
  'Pizza with a unique topping combination of pineapple and ham',
  'White bathtub with a shower curtain ready for a soak',
  'Fishing rod propped against a dock'],
 'label': [0, 1, 1, 0, 0]}